In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, precision_score, recall_score
from lightgbm import LGBMClassifier
import joblib

train = pd.read_csv("train.csv")
# --------------------------
# 1. Признаки / таргеты
# --------------------------
feat_cols = [c for c in train.columns if str(c).startswith('v_')]
X = train[feat_cols]
y_month = train['month_churn']
y_week  = train['week_churn']

# --------------------------
# 2. Time-based split
# --------------------------
train['report_date'] = pd.to_datetime(train['report_date'], errors='coerce')

val_cut = pd.to_datetime('2025-01-01')

train_mask = train['report_date'] < val_cut
val_mask   = train['report_date'] >= val_cut

X_tr, X_val = X.loc[train_mask], X.loc[val_mask]
ytr_month, yval_month = y_month.loc[train_mask], y_month.loc[val_mask]
ytr_week,  yval_week  = y_week.loc[train_mask],  y_week.loc[val_mask]


# --------------------------
# 3. Общие параметры LGBM (валидные)
# --------------------------
lgb_params = {
    # model params (exist in sklearn wrapper)
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "max_depth": -1,
    "learning_rate": 0.05,
    "n_estimators": 2000,           # large, но используем раннюю остановку
    "subsample_for_bin": 200000,
    "objective": "binary",
    "class_weight": "balanced",     # учёт дисбаланса
    "min_split_gain": 0.0,
    "min_child_weight": 1e-3,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "random_state": 42,
    "n_jobs": -1,
    "importance_type": "gain"
}

# --------------------------
# 4. Train model for month_churn
# --------------------------
model_month = LGBMClassifier(**lgb_params)

model_month.fit(
    X_tr, ytr_month,
    eval_set=[(X_val, yval_month)],
    eval_metric='auc'
)

# Predict & metric
pval_month = model_month.predict_proba(X_val)[:,1]
auc_month = roc_auc_score(yval_month, pval_month)
print(f"AUC month_churn: {auc_month:.4f}")

# --------------------------
# 5. Train model for week_churn
# --------------------------
model_week = LGBMClassifier(**lgb_params)

model_week.fit(
    X_tr, ytr_week,
    eval_set=[(X_val, yval_week)],
    eval_metric='auc'
)

pval_week = model_week.predict_proba(X_val)[:,1]
auc_week = roc_auc_score(yval_week, pval_week)
print(f"AUC week_churn: {auc_week:.4f}")

[LightGBM] [Info] Number of positive: 26290, number of negative: 159120
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.106794 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51844
[LightGBM] [Info] Number of data points in the train set: 185410, number of used features: 271
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
AUC month_churn: 0.8038
[LightGBM] [Info] Number of positive: 48377, number of negative: 137033
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 51844
[LightGBM] [Info] Number of data points in the train set: 185410, number of used features: 271
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.50

In [4]:
test_df = pd.read_csv("test.csv")

# Получаем признаки для test_df
X_test = test_df[feat_cols]

# Предсказания вероятностей для month_churn
month_proba = model_month.predict_proba(X_test)[:, 1]

# Предсказания вероятностей для week_churn
week_proba = model_week.predict_proba(X_test)[:, 1]

# Создаем результирующий DataFrame
result_df = pd.DataFrame({
    'nid': test_df['nid'].values,
    'week_churn': week_proba,
    'month_churn': month_proba
})

# Сохраняем в CSV
result_df.to_csv('predictions.csv', index=False)

# Проверяем первые несколько строк
print(result_df.head(20))

      nid  week_churn  month_churn
0    2378    0.207432     0.020691
1    4004    0.462347     0.780223
2    9151    0.008736     0.004277
3    6701    0.229042     0.009769
4   10771    0.022968     0.001005
5    6641    0.264421     0.030674
6    7581    0.294635     0.036402
7   10462    0.015892     0.051361
8   10982    0.016562     0.000942
9   10406    0.079355     0.042641
10   6283    0.271502     0.225246
11   7594    0.014906     0.002215
12  10135    0.032170     0.255073
13   8118    0.049225     0.006415
14  11247    0.048485     0.018591
15   1646    0.805299     0.752025
16   7418    0.044240     0.116329
17   1202    0.069272     0.064593
18  11127    0.279902     0.062107
19  10790    0.116791     0.001576
